# PyAge temporal example (Ploemeur)

This notebook reproduces the temporal calibration workflow used in
`scripts/launcher_temporal.py`, but in a step-by-step and didactic way.
It is designed to be readable by a scientific audience and to support
publication-quality explanations.

## Scientific context (short)
We calibrate Lumped Parameter Models (LPMs) against a multi-date
concentration dataset in order to reconstruct temporal concentration
chronicles for a monitoring well. The workflow combines:
- data ingestion and basic quality handling,
- model configuration (LPM family, sampling, priors),
- Metropolis-Hastings calibration,
- visualization of chronicle fits and parameter distributions.

## What you should expect
- intermediate summaries of the dataset (tracer list, dates, statistics),
- a clear, reproducible configuration loaded from YAML,
- inline figures during calibration (chronicles + parameter distributions),
- a results folder with saved outputs (tables + PNGs).

Tip: run the notebook from top to bottom once to populate results, then
re-run only the analysis and plotting cells as needed.


In [ ]:
import sys
from pathlib import Path
import os

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'pyproject.toml').exists() and (parent / 'pyage').exists():
        ROOT = parent
        break

print('CWD:', os.getcwd())
print('ROOT:', ROOT)
print('Has pyage:', (ROOT / 'pyage').exists())
print('Has data_core:', (ROOT / 'data_core').exists())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('sys.path[0]:', sys.path[0])


## Environment and reproducibility
We print paths and confirm the repository root so the notebook is portable.


## Inline figures
We force inline Matplotlib output so that figures appear directly in the
notebook (as in a paper figure panel).


In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
import matplotlib.pyplot as plt
set_matplotlib_formats("png")


## Imports (matching launcher_temporal.py)
We keep the same core imports as the CLI launcher for transparency and
reproducibility, with a few extras for concise reporting.


In [ ]:
import yaml
from pydantic import ValidationError
from pprint import pprint

import numpy as np
from IPython.display import display, Image

import pyage.global_parameters as gp
import pyage.concentrations.concentrations as co
from pyage.concentrations import concentrations_time as ct
import pyage.calibration.utils.calibration_core as calbas
import pyage.calibration.methods.metropolis_hastings as cMH

from pyage.config.models import (
    TemporalCalibrationCfg,
    TemporalDatasetCfg,
    TemporalFiguresCfg,
    TemporalLpmModelsCfg,
    TemporalParams,
    TemporalResultsCfg,
    TemporalWorkflowCfg,
    TEMPORAL_VALID_MODES,
)


## Parameters
All settings are stored in a YAML file. This makes the workflow explicit
and reproducible across machines.


In [ ]:
params_path = ROOT / 'examples' / 'ploemeur_temporal' / 'ploemeur_temporal.yaml'
print('Params:', params_path)


### Configuration preview (YAML)
A short preview of the YAML file used for the run.


In [ ]:
print(params_path.read_text(encoding='utf-8'))


## Step 1 - Load and validate YAML configuration
We parse the YAML into typed Pydantic models to catch missing fields or
invalid values early.


In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f'Missing params file: {path}')
    with path.open('r', encoding='utf-8') as handle:
        return yaml.safe_load(handle) or {}

def load_params_validated(path: Path) -> TemporalParams:
    data = load_yaml(path)
    try:
        return TemporalParams.model_validate(data)
    except ValidationError as exc:
        raise ValueError(f"Invalid launcher_temporal config:\n{exc}") from exc

params = load_params_validated(params_path)
dataset_cfg: TemporalDatasetCfg = params.dataset
cal_cfg: TemporalCalibrationCfg = params.calibration
figures_cfg: TemporalFiguresCfg = params.figures
workflow_cfg: TemporalWorkflowCfg = params.workflow
lpm_cfg: TemporalLpmModelsCfg = params.lpm_models
results_cfg: TemporalResultsCfg = params.results


### Configuration summary (typed models)
We show the validated configuration as dictionaries.


In [ ]:
print("Dataset config:")
pprint(dataset_cfg.model_dump())
print("\nWorkflow config:")
pprint(workflow_cfg.model_dump())
print("\nCalibration config:")
pprint(cal_cfg.model_dump())
print("\nFigures config:")
pprint(figures_cfg.model_dump())
print("\nLPM models config:")
pprint(lpm_cfg.model_dump())
print("\nResults config:")
pprint(results_cfg.model_dump())


## Step 2 - Resolve paths and load dataset
We resolve repository-relative paths, load the concentrations table, and
apply the configured error model.


In [ ]:
def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    if not path.is_absolute():
        path = ROOT / path
    return path

dataset_file = dataset_cfg.file
dataset_path = resolve_path(dataset_file)
if not dataset_path.exists():
    raise FileNotFoundError(f'Dataset file not found: {dataset_path}')

mode = workflow_cfg.mode
if mode not in TEMPORAL_VALID_MODES:
    raise ValueError(f'workflow.mode must be one of {sorted(TEMPORAL_VALID_MODES)}')

lpm_list = lpm_cfg.list or ["exp_shifted", "ig", "ig_shifted"]
lpm_directory = lpm_cfg.directory or str(gp.DIRECTORY_LPM_DATA)
lpm_directory_path = resolve_path(lpm_directory)
if not lpm_directory_path.exists():
    raise ValueError(f'lpm_models.directory does not exist: {lpm_directory_path}')

def results_root(results_cfg: TemporalResultsCfg) -> Path:
    if results_cfg.use_default:
        return Path(gp.ROOT_DIRECTORY_RESULTS)
    if not results_cfg.directory:
        raise ValueError('results.directory must be set when use_default is false.')
    out = resolve_path(results_cfg.directory)
    out.mkdir(parents=True, exist_ok=True)
    return out

results_root_path = results_root(results_cfg)

# Load concentrations from file
cdata = co.Concentrations(file_load=True, file_name=str(dataset_path))
if dataset_cfg.error_rel is not None:
    cdata.error_affect_from_value(dataset_cfg.error_rel)

print('Loaded dates:', cdata.cv['date'].unique())
print('LPM list:', lpm_list)
print('Results root:', results_root_path)


### Data characterization
We inspect the raw concentration table, basic statistics, and the temporal
coverage of the observations.


In [ ]:
# Quick look at the raw concentrations table
# (pandas DataFrame stored in cdata.cv)
display(cdata.cv.head(10))

print("Observations:", len(cdata.cv))
print("Tracers:", sorted(cdata.cv["element"].unique()))
print("Unique dates:", cdata.cv["date"].nunique())
print("Date range:", (cdata.cv["date"].min(), cdata.cv["date"].max()))

# Counts per tracer
display(
    cdata.cv["element"]
    .value_counts()
    .rename_axis("tracer")
    .to_frame("n_obs")
)

# Basic stats for numeric columns
# (date, concentration, error)
display(cdata.cv[["date", "concentration", "error"]].describe())
print("Zero errors:", int((cdata.cv["error"] == 0).sum()))


### Data overview plot (observations only)
Scatter plots by tracer, before any model calibration.


In [ ]:
tracers = sorted(cdata.cv["element"].unique())
if tracers:
    ncols = 2
    nrows = int(np.ceil(len(tracers) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.5 * nrows), squeeze=False)
    for ax, tracer in zip(axs.flatten(), tracers):
        df = cdata.cv[cdata.cv["element"] == tracer]
        ax.scatter(df["date"], df["concentration"], s=20)
        ax.set_title(tracer)
        ax.set_xlabel("Year")
        ax.set_ylabel("Concentration")
    for ax in axs.flatten()[len(tracers):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


### Optional: inspect one LPM parameterization
This is a light, optional check to see parameter names and ranges for the
first LPM in the list.


In [ ]:
import pyage.lpm.lpm_build as lpm_build_module

example_lpm = lpm_build_module.lpm_build(lpm_list[0], lpm_directory_path)
print("Example LPM:", example_lpm.name)
print("Parameters:", example_lpm.get_param_names())
for name in example_lpm.get_param_names():
    pmin = example_lpm.get_p_min(name)
    pmax = example_lpm.get_p_max(name)
    print(f"{name}: [{pmin}, {pmax}]")


## Step 3 - Calibration helpers
These functions mirror the launcher: prepare display options, build the
Metropolis-Hastings config, and run a single calibration.


In [ ]:
def prepare_display(output_dir: Path, figures_cfg: TemporalFiguresCfg) -> gp.display_options:
    display = gp.display_options()
    display.text = False
    display.figure = bool(figures_cfg.temporal or figures_cfg.distributions)
    # Notebook: keep figures open and inline
    display.figure_save = False
    display.figure_close = False
    display.directory = str(output_dir)
    return display

def build_mh_config(cal_cfg: TemporalCalibrationCfg, lpm_number: int) -> cMH.MHConfig:
    seed_enabled = cal_cfg.seed_enabled
    seed_value = cal_cfg.seed if seed_enabled else None
    mh_kwargs = {}
    if seed_enabled:
        mh_kwargs['seed'] = seed_value
    return cMH.MHConfig(
        nstep=int(cal_cfg.mh_nsteps),
        burn_in=float(cal_cfg.burn_in),
        nskip=int(cal_cfg.nskip),
        prior_option=True,
        prior_type='parametric',
        likelihood=True,
        monitor=False,
        display_traj=False,
        display_text=False,
        lpm_number=lpm_number,
        **mh_kwargs,
    )

def run_calibration(
    cdata: co.Concentrations,
    lpm_type: str,
    output_dir: Path,
    lpm_directory: Path,
    cal_cfg: TemporalCalibrationCfg,
    figures_cfg: TemporalFiguresCfg,
    mode: str,
):
    display = prepare_display(output_dir, figures_cfg)

    # Calibration resolution and sampling controls
    explo_res = int(cal_cfg.explo_res)
    mh_nsteps = int(cal_cfg.mh_nsteps)
    lpm_number = int(cal_cfg.lpm_number)
    if lpm_number <= 0:
        lpm_number = max(min(int(mh_nsteps / 50), 5000), 10)

    calib_basis = calbas.CalibrationCore(
        cdata,
        lpm_type,
        display_options=display,
        directory_lpm=str(lpm_directory),
        nmodels=explo_res,
        reachconc=False,
    )
    calib_basis.prepare()

    mh_config = build_mh_config(cal_cfg, lpm_number)
    calstrat = cMH.MetropolisHastings(config=mh_config)
    calstrat.MH_step.define_by_value()
    calstrat.update_calibbasis(calib_basis)
    lpm_results = calstrat.perform()
    calstrat.write_calibrated_lpm(lpm_results)

    if figures_cfg.temporal:
        ct.display_concentration_chronicles(
            cdata,
            lpm_results,
            calstrat.method,
            display,
            time_span_mode=mode,
            lpm_number=lpm_number,
        )
        plt.show()

    if figures_cfg.distributions:
        lpm_results.display_parameters_dist(
            self_method=calstrat.method,
            directory=None,
        )
        plt.show()
        if figures_cfg.concentrations_2d:
            lpm_results.display_concentrations_dist(
                self_method=calstrat.method,
                concentrations_reference=cdata,
                directory=None,
            )
            plt.show()

    return lpm_results


## Step 4 - Main execution
Run the temporal calibration over the configured LPM list. Depending on
settings, this can take a few minutes.


In [ ]:
dataset_stem = Path(dataset_file).stem
mode = workflow_cfg.mode

output_base = results_root_path / 'ploemeur_temporal' / dataset_stem / mode
output_base.mkdir(parents=True, exist_ok=True)

if mode == 'span':
    span_dir = output_base / 'span_full'
    span_dir.mkdir(parents=True, exist_ok=True)
    for lpm_type in lpm_list:
        lpm_dir = span_dir / lpm_type
        lpm_dir.mkdir(parents=True, exist_ok=True)
        print('Running span:', lpm_type, lpm_dir)
        run_calibration(
            cdata=cdata,
            lpm_type=lpm_type,
            output_dir=lpm_dir,
            lpm_directory=lpm_directory_path,
            cal_cfg=cal_cfg,
            figures_cfg=figures_cfg,
            mode=mode,
        )

else:
    raise ValueError(f'Invalid workflow mode: {mode}')

print('Results root:', output_base)


## Post-run outputs (quick check)
List saved PNGs and display a couple of them if available.


In [ ]:
print("Results root:", output_base)

pngs = sorted(Path(output_base).rglob("*.png"))
print("PNG files:", len(pngs))
for p in pngs[:5]:
    print(p)

# Display a couple of figures if present
for p in pngs[:2]:
    display(Image(filename=str(p)))
